# 03 · Continuous training (CT)

CI asks *is the code correct?* on every push. CD ships it once green. **Continuous training** asks a different question on a cadence: *as new data and drift arrive, is a retrained candidate better, and safe to promote?*

| Loop | Trigger | Question | Where |
| --- | --- | --- | --- |
| **CI** | every push | is the code correct? | `.github/workflows/ci.yml` |
| **CD** | CI green | is it deployed? | the promotion gate |
| **CT** | schedule / drift / new labeled data | is a retrained candidate better and safe? | `.github/workflows/ct.yml` |

"Training" here is not gradient descent on weights (the LLM is a hosted Groq model). It is the data-and-prompt layer this project *does* own: the retrieval index, the governed enrichment features, and the router and answer prompts. This notebook runs the CT **policy** offline (no keys, no Docker) from `mlops/ct.py`.

In [ ]:
# Bootstrap: make the repo importable no matter where the kernel starts. Run this cell first.
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
from mlops.ct import evaluate_trigger, decide_promotion, run_ct_cycle
print("CT policy loaded from mlops/ct.py")

## 1. When does CT run?

CT does not run on every push (that is CI). It fires only when something changed: the weekly schedule, measured **drift** over threshold, or enough new **human-verified answers** from the review-queue flywheel. `evaluate_trigger` returns the decision *and* the reasons, so every run is self-documenting.

In [ ]:
cases = [
    ('drift over threshold', dict(drift_score=2.0, drift_threshold=1.0, new_labeled=0, min_new_labeled=10)),
    ('enough new labeled data', dict(drift_score=0.0, drift_threshold=1.0, new_labeled=25, min_new_labeled=10)),
    ('weekly cadence', dict(drift_score=None, drift_threshold=1.0, new_labeled=0, min_new_labeled=10, scheduled=True)),
    ('nothing changed', dict(drift_score=0.0, drift_threshold=1.0, new_labeled=3, min_new_labeled=10)),
]
for label, c in cases:
    fired, reasons = evaluate_trigger(**c)
    print((' FIRE ' if fired else ' quiet'), '|', label, '->', reasons or '(no trigger)')

## 2. What gets promoted?

A retrained candidate is promoted **only** when it clears every bar: it beats the baseline by a margin on a held-out split, the regression gate is green, and the safety check passed. A missing score, a failed gate, or a failed safety check is an automatic no, so an un-evaluated or unsafe (reward-hacking) candidate can never ship.

In [ ]:
rows = [
    ('beats baseline, gate + safety green', dict(baseline_score=0.80, candidate_score=0.86, min_gain=0.01, gate_passed=True, safety_passed=True)),
    ('gain below the margin', dict(baseline_score=0.80, candidate_score=0.805, min_gain=0.01, gate_passed=True, safety_passed=True)),
    ('gate regressed', dict(baseline_score=0.80, candidate_score=0.90, min_gain=0.01, gate_passed=False, safety_passed=True)),
    ('failed safety (reward-hacking)', dict(baseline_score=0.80, candidate_score=0.95, min_gain=0.01, gate_passed=True, safety_passed=False)),
]
for label, kw in rows:
    print(('PROMOTE' if decide_promotion(**kw) else 'keep   '), '|', label)

## 3. One CT cycle, end to end

`run_ct_cycle` ties it together over injected steps (so the policy is testable offline): a `train()` step that retrains and returns the scores, and a `gate()` step. By default CT **proposes** a promotion and a human approves it; nothing self-deploys. `--auto-promote` is opt-in.

In [ ]:
train = lambda: {'baseline_score': 0.80, 'candidate_score': 0.86, 'safety_passed': True,
                 'candidate_path': 'mlops/prompt_registry/tiebreak_system.candidate.json'}
gate = lambda: {'passed': True, 'score': 1.0}

proposed = run_ct_cycle(trigger_fired=True, reasons=['scheduled cadence'], train=train, gate=gate)
print('default (propose):  recommended=%s  promoted=%s' % (proposed.promote_recommended, proposed.promoted))
for n in proposed.notes: print('   -', n)

auto = run_ct_cycle(trigger_fired=True, reasons=['forced'], train=train, gate=gate, auto_promote=True)
print('auto-promote:       recommended=%s  promoted=%s' % (auto.promote_recommended, auto.promoted))

## 4. The real cycle

`make ct` (see `scripts/run_ct.py`) wires the real signals: drift from the recorded trace store, new labeled data from the review-queue flywheel, the safety-gated OPRO prompt-optimization loop as the training step, and the same eval gate CI runs. It writes `evaluation/reports/ct_report.json` and logs the cycle to MLflow. The weekly `.github/workflows/ct.yml` uploads that report as the promotion proposal for a human to approve. If a cycle has run, its report loads below.

In [ ]:
import json, os
p = 'evaluation/reports/ct_report.json'
if os.path.exists(p):
    r = json.load(open(p))
    print('last CT cycle:', 'TRIGGERED' if r['triggered'] else 'no trigger')
    print('  reasons:', r['reasons'])
    print('  baseline -> candidate:', r['baseline_score'], '->', r['candidate_score'])
    print('  gate_passed:', r['gate_passed'], '| recommended:', r['promote_recommended'], '| promoted:', r['promoted'])
else:
    print('no ct_report.json yet; run `make ct` to produce one')